# Deduplication

In ETL and data engineering, you frequently encounter data where the same entity appears with slightly different spellings -- names, addresses, product descriptions, company names, and so on. Exact string matching fails in these cases, so we need **fuzzy matching**: techniques that measure how similar two strings are, even when they are not identical.

The [`thefuzz`](https://github.com/seatgeek/thefuzz) library implements fuzzy string matching based on **Levenshtein distance** -- the minimum number of single-character edits (insertions, deletions, or substitutions) required to change one string into another. It provides several matching strategies and a convenient `process` module for matching against lists of candidates.

In this notebook we will cover:
1. Levenshtein Distance & Simple Ratio
2. Partial Ratio & Token Sort Ratio
3. The Process module
4. A practical deduplication exercise with pandas

Run the following cell to install and import the required library.

In [ ]:
%pip install thefuzz[speedup]

In [ ]:
from thefuzz import fuzz, process

## Section 1: Levenshtein Distance & Simple Ratio

The function `fuzz.ratio()` computes a similarity score between two strings on a scale from **0 to 100**, where 100 means the strings are identical. Under the hood, it calculates the Levenshtein distance and normalizes it by the total length of both strings.

```python
### CODE EXAMPLE
from thefuzz import fuzz

# Identical strings score 100
fuzz.ratio("hello", "hello")      # 100

# Small differences lower the score
fuzz.ratio("hello", "hallo")      # 80

# Completely different strings score low
fuzz.ratio("hello", "world")      # 20
```

The simple ratio is case-sensitive, so `"Python"` and `"python"` will not score 100.

### Exercise 1

Compare the following pairs of strings using `fuzz.ratio()` and print the score for each pair:
- `("Python", "Python")`
- `("Python", "python")`
- `("Python", "Pythonic")`
- `("Python", "Java")`

**Explanation:** We call `fuzz.ratio()` on each pair to compute the Levenshtein-based similarity score. Identical strings score 100. The case difference in `"Python"` vs `"python"` causes a small penalty because `fuzz.ratio()` is case-sensitive. `"Pythonic"` shares most characters with `"Python"` so it scores relatively high, while `"Java"` is completely different and scores low.

In [ ]:
### FILL IN; START

pairs = [
    ("Python", "Python"),
    ("Python", "python"),
    ("Python", "Pythonic"),
    ("Python", "Java")
]

for s1, s2 in pairs:
    score = fuzz.ratio(s1, s2)
    print(f"fuzz.ratio('{s1}', '{s2}') = {score}")

### FILL IN; END

### Exercise 2

Compare `"Wortell Smart Learning"` with `"Wortell Smart Learning B.V."` using `fuzz.ratio()`.

Then compare `"Wortell Smart Learning BV"` with `"Wortell Smart Learning B.V."`.

Print both scores. Why do the scores differ? Think about what characters contribute to the Levenshtein distance in each case.

**Explanation:** The first comparison has a larger length difference (the second string has the extra " B.V." suffix), which results in a lower score. The second comparison is between two strings that are much closer in length -- `"BV"` vs `"B.V."` only differs by the dots and a space -- so the score is higher. This illustrates that `fuzz.ratio()` is sensitive to differences in string length: extra characters dilute the overall similarity score.

In [ ]:
### FILL IN; START

score1 = fuzz.ratio("Wortell Smart Learning", "Wortell Smart Learning B.V.")
print(f"'Wortell Smart Learning' vs 'Wortell Smart Learning B.V.' = {score1}")

score2 = fuzz.ratio("Wortell Smart Learning BV", "Wortell Smart Learning B.V.")
print(f"'Wortell Smart Learning BV' vs 'Wortell Smart Learning B.V.' = {score2}")

print(f"\nThe second pair scores higher ({score2} vs {score1}) because the strings are closer in length.")
print("The dots in 'B.V.' only add a small edit distance compared to 'BV', whereas")
print("the first pair has a much larger length difference (missing ' B.V.' entirely).")

### FILL IN; END

## Section 2: Partial Ratio & Token Sort Ratio

The simple ratio works well when strings are roughly the same length, but it can produce low scores when one string is a substring of the other, or when words appear in a different order. `thefuzz` provides two additional methods to handle these situations:

- **`fuzz.partial_ratio()`** -- Finds the best matching substring. It slides the shorter string across the longer string and returns the highest ratio found. This is useful when one string contains extra text (e.g. a suffix like "B.V." or "Inc.").

- **`fuzz.token_sort_ratio()`** -- Splits both strings into tokens (words), sorts them alphabetically, and then computes the ratio. This handles cases where the same words appear in a different order.

```python
### CODE EXAMPLE
from thefuzz import fuzz

# partial_ratio: "Learning" is fully contained in the longer string
fuzz.partial_ratio("Wortell Smart Learning", "Wortell Smart Learning B.V.")  # 100

# token_sort_ratio: handles reordered words
fuzz.token_sort_ratio("John Smith", "Smith John")  # 100
```

### Exercise 3

Compare `"New York City"` and `"City of New York"` using all three methods:
- `fuzz.ratio()`
- `fuzz.partial_ratio()`
- `fuzz.token_sort_ratio()`

Print the results for each method. Which method handles the word reordering best?

**Explanation:** We apply all three matching strategies to the same pair to illustrate their differences. `fuzz.ratio()` gives a moderate score because the character sequences differ due to word reordering. `fuzz.partial_ratio()` may score higher if substrings align well. `fuzz.token_sort_ratio()` handles this best because it sorts the tokens alphabetically before comparing, effectively neutralizing the word order difference. The extra word `"of"` in the second string prevents a perfect score even with token sorting.

In [ ]:
### FILL IN; START

s1 = "New York City"
s2 = "City of New York"

r1 = fuzz.ratio(s1, s2)
r2 = fuzz.partial_ratio(s1, s2)
r3 = fuzz.token_sort_ratio(s1, s2)

print(f"fuzz.ratio:            {r1}")
print(f"fuzz.partial_ratio:    {r2}")
print(f"fuzz.token_sort_ratio: {r3}")
print(f"\ntoken_sort_ratio handles word reordering best with a score of {r3}.")

### FILL IN; END

### Exercise 4

Given the following list of company names:

```python
companies = [
    "Microsoft Corporation",
    "Microsoft Corp.",
    "Microsoft Corp",
    "Apple Inc.",
    "Apple Inc",
    "Apple Incorporated",
    "Google LLC",
    "Alphabet Inc."
]
```

Find and print all pairs of company names where `fuzz.ratio() > 80`. These are potential duplicates that refer to the same company.

**Explanation:** We use a nested loop to compare every unique pair of company names. The inner loop starts at `i + 1` to avoid comparing a name with itself and to avoid duplicate pairs (A vs B and B vs A). We only print pairs where `fuzz.ratio()` exceeds the threshold of 80. This brute-force approach has O(n^2) complexity and works fine for small lists, but for large datasets you would want to use blocking or indexing strategies.

In [ ]:
companies = [
    "Microsoft Corporation",
    "Microsoft Corp.",
    "Microsoft Corp",
    "Apple Inc.",
    "Apple Inc",
    "Apple Incorporated",
    "Google LLC",
    "Alphabet Inc."
]

### FILL IN; START

for i in range(len(companies)):
    for j in range(i + 1, len(companies)):
        score = fuzz.ratio(companies[i], companies[j])
        if score > 80:
            print(f"{companies[i]:25s} <-> {companies[j]:25s}  score: {score}")

### FILL IN; END

## Section 3: Process Module

When you need to match a string against a list of candidates, the `process` module provides convenient functions:

- **`process.extract(query, choices, limit=N)`** -- Returns the top N best matches from the list, each as a tuple of `(match, score)`.
- **`process.extractOne(query, choices)`** -- Returns only the single best match as a tuple of `(match, score)`.

These are particularly useful for **standardizing data against a master list** -- for example, mapping messy user-entered city names to a canonical set of city names.

```python
### CODE EXAMPLE
from thefuzz import process

choices = ["Atlanta Falcons", "New York Jets", "New York Giants", "Dallas Cowboys"]

# Get top 2 matches
process.extract("new york jets", choices, limit=2)
# [('New York Jets', 100), ('New York Giants', 79)]

# Get only the best match
process.extractOne("cowboys", choices)
# ('Dallas Cowboys', 90)
```

### Exercise 5

Given the following master list:

```python
choices = ["Atlanta Falcons", "New York Jets", "New York Giants", "Dallas Cowboys"]
```

Use `process.extractOne()` to find the best match for each of the following queries:
- `"new york jets"`
- `"cowboys dallas"`
- `"falcons"`

Print the result (match and score) for each query.

**Explanation:** `process.extractOne()` returns a tuple of `(best_match, score)` representing the closest match from the choices list. It handles case differences automatically ("new york jets" matches "New York Jets"). For "cowboys dallas" the word order is reversed, but the default scorer still finds the right match because the character overlap is high. This is a quick way to standardize free-text input against a known set of values.

In [ ]:
choices = ["Atlanta Falcons", "New York Jets", "New York Giants", "Dallas Cowboys"]

### FILL IN; START

queries = ["new york jets", "cowboys dallas", "falcons"]

for query in queries:
    result = process.extractOne(query, choices)
    print(f"Query: '{query}' -> Best match: '{result[0]}', Score: {result[1]}")

### FILL IN; END

### Exercise 6

You have a list of messy city names from user input and a master list of correct city names:

```python
messy_names = ["Amstrdam", "amsterdam", "Rotterdam", "Roterdam", "Den Haag", "den haag", "The Hague", "Utrecht", "Utrect"]
master_list = ["Amsterdam", "Rotterdam", "Den Haag", "Utrecht"]
```

Write code to map each messy name to its best match from the master list using `process.extractOne()`. Only accept matches with a **score threshold of 80** or higher. Print the mapping (messy name -> best match with score).

**Explanation:** We iterate over each messy name and use `process.extractOne()` with `score_cutoff=80` to find the best match above our threshold. When no match exceeds the threshold, `extractOne` returns `None`, so we check for that case. This is useful in real data cleaning pipelines where you want to automatically standardize known variations but flag truly unknown values for manual review. Note that `"The Hague"` may not match `"Den Haag"` because they are translations rather than misspellings.

In [ ]:
messy_names = ["Amstrdam", "amsterdam", "Rotterdam", "Roterdam", "Den Haag", "den haag", "The Hague", "Utrecht", "Utrect"]
master_list = ["Amsterdam", "Rotterdam", "Den Haag", "Utrecht"]

### FILL IN; START

for name in messy_names:
    result = process.extractOne(name, master_list, score_cutoff=80)
    if result:
        print(f"'{name}' -> '{result[0]}' (score: {result[1]})")
    else:
        print(f"'{name}' -> No match above threshold")

### FILL IN; END

## Section 4: Practical Deduplication Exercise

In real-world data pipelines, deduplication often involves working with DataFrames. The typical workflow is:
1. Identify potential duplicate pairs using fuzzy matching.
2. Decide on a canonical (standardized) name for each group of duplicates.
3. Map all variations to the canonical name.

Let's put everything together in a practical exercise.

### Exercise 7

Create a pandas DataFrame with a column `"customer_name"` containing the following names:

```python
["John Smith", "Jon Smith", "John Smit", "Jane Doe", "Janet Doe", "Jane D.", "Bob Wilson", "Robert Wilson", "Bob Willson"]
```

Write a function `find_duplicates(names, threshold=80)` that takes a list of names and returns a list of tuples `(name1, name2, score)` for all pairs where `fuzz.ratio(name1, name2) > threshold`.

Apply this function to the customer names and print the potential duplicate pairs.

**Explanation:** We define a reusable function that iterates over all unique pairs of names (using the `i + 1` trick to avoid duplicate comparisons) and collects those that exceed the similarity threshold. The function returns a list of tuples, making it easy to inspect or further process the results. We then create a DataFrame and apply the function to its `customer_name` column converted to a list.

In [ ]:
import pandas as pd

customer_names = ["John Smith", "Jon Smith", "John Smit", "Jane Doe", "Janet Doe", "Jane D.", "Bob Wilson", "Robert Wilson", "Bob Willson"]

### FILL IN; START

df = pd.DataFrame({"customer_name": customer_names})

def find_duplicates(names, threshold=80):
    duplicates = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            score = fuzz.ratio(names[i], names[j])
            if score > threshold:
                duplicates.append((names[i], names[j], score))
    return duplicates

duplicate_pairs = find_duplicates(df["customer_name"].tolist())

print("Potential duplicate pairs:")
for name1, name2, score in duplicate_pairs:
    print(f"  '{name1}' <-> '{name2}' (score: {score})")

### FILL IN; END

### Exercise 8

Extend the previous exercise: given the DataFrame with customer names, write code to create a new column `"canonical_name"` where each name is mapped to a standardized version.

Approach:
1. Start with the list of unique customer names.
2. For each name, use `process.extractOne()` against the list of unique names to find its best match.
3. Build groups of similar names and pick one canonical name per group (e.g., the first name encountered).
4. Add the `"canonical_name"` column to the DataFrame.

Print the resulting DataFrame showing both the original and canonical names.

**Explanation:** We use a Union-Find (disjoint set) approach to group similar names together. For each name, we compare it against all other names. When two names have a `fuzz.ratio` above the threshold, we link them into the same group by tracing back to a "canonical" representative. We use a dictionary to map each name to its canonical form, always picking the first name encountered in a group as the representative. Finally, we apply this mapping to the DataFrame using `.map()`. This approach handles transitive similarity: if A matches B and B matches C, all three end up with the same canonical name.

In [ ]:
import pandas as pd

customer_names = ["John Smith", "Jon Smith", "John Smit", "Jane Doe", "Janet Doe", "Jane D.", "Bob Wilson", "Robert Wilson", "Bob Willson"]
df = pd.DataFrame({"customer_name": customer_names})

### FILL IN; START

threshold = 80
names = df["customer_name"].tolist()

# Build a mapping from each name to its canonical form
canonical_map = {}

for name in names:
    if name not in canonical_map:
        # This name starts a new group; it is its own canonical form
        canonical_map[name] = name

    # Compare against all other names not yet assigned to this group
    for other_name in names:
        if other_name not in canonical_map:
            score = fuzz.ratio(name, other_name)
            if score > threshold:
                # Assign to the same canonical name
                canonical_map[other_name] = canonical_map[name]

df["canonical_name"] = df["customer_name"].map(canonical_map)

print(df.to_string(index=False))

### FILL IN; END

## Summary

In this notebook we explored fuzzy string matching with `thefuzz`:

| Method | Best used when... |
|---|---|
| `fuzz.ratio()` | Strings are roughly the same length with minor spelling differences |
| `fuzz.partial_ratio()` | One string is a substring of the other (e.g., missing suffixes like "Inc." or "B.V.") |
| `fuzz.token_sort_ratio()` | Words are the same but appear in a different order |
| `process.extract()` | You need the top N matches from a list of candidates |
| `process.extractOne()` | You need only the single best match from a list |

**Limitations to keep in mind:**
- Fuzzy matching is purely character-based -- it does not understand meaning. Two semantically similar strings (e.g., "The Hague" and "Den Haag") may score low.
- Performance degrades with very long strings or very large candidate lists. For large-scale deduplication, consider indexing or blocking strategies to reduce the number of comparisons.
- Threshold selection is critical: too low means false positives, too high means missed duplicates. Always validate with domain knowledge.